In [1]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import torch.nn as nn

In [5]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model=torchvision.models.mobilenet_v3_small(weights="IMAGENET1K_V1")
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)

model.load_state_dict(torch.load("../models/best_model.pth",map_location=device))
model=model.to(device)
print("Model loaded successfully!")

Model loaded successfully!


In [6]:
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_dataset = datasets.ImageFolder(root="../data", transform=val_transforms)
class_names  = full_dataset.classes   # ['fire', 'no_fire']

total      = len(full_dataset)
train_size = int(0.8 * total)
val_size   = total - train_size

torch.manual_seed(42)   # same seed = same split as training
_, val_dataset = random_split(full_dataset, [train_size, val_size])

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Validation images: {len(val_dataset)}")
print(f"Classes: {class_names}")

Validation images: 200
Classes: ['fire', 'no_fire']


In [7]:
all_preds  = []
all_labels = []
all_probs  = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

correct = (all_preds == all_labels).sum()
total   = len(all_labels)
print(f"Accuracy: {correct}/{total} = {100*correct/total:.2f}%")

Accuracy: 199/200 = 99.50%


In [8]:
print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names
))

              precision    recall  f1-score   support

        fire       1.00      0.99      1.00       153
     no_fire       0.98      1.00      0.99        47

    accuracy                           0.99       200
   macro avg       0.99      1.00      0.99       200
weighted avg       1.00      0.99      1.00       200

